# NB02 — Schema Design & Data Ingestion
**Signal/Pulse · Database Layer**

Ingests all NB01 raw data into a normalised SQLite database.

**Sources and file paths:**
- Rakuten    : `data/raw/rakuten/products/rakuten_products_{genre_id}_{date}.json`
- @cosme L1  : `data/raw/cosme/products/cosme_products_{cat_id}.json`
- @cosme L2  : `data/raw/cosme/reviews/cosme_reviews_{cat_id}_{pid}.json`
- Amazon L1  : `data/raw/amazon/products/amazon_products.json`
- Amazon L2  : `data/raw/amazon/reviews/amazon_reviews_{asin}.json`
- YouTube    : `data/raw/youtube/youtube_video_stats.json` + `comments/youtube_comments_{vid}.json`
- Trends     : `data/raw/trends/trends_A_*.json`, `trends_B_*.json`, `trends_C_*.json`

**Key design decisions:**
- All review sources share one `reviews` table, distinguished by `source_id`
- `review_year` / `review_month` stored as integers
- `UNIQUE(source_id, source_review_id)` — idempotent, safe to re-run
- Rakuten goes into `products` not `reviews` — aggregate data, not individual reviews
- YouTube `search_category` preserved from tiered scrape
- Block C Rising queries stored as raw JSON — loaded in NB06, not time-series

**Output:** `data/signal_pulse.db`

## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import sqlite3
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

from src.utils import load_env, DATA_RAW
from src.config import (
    print_ingestion_summary,
    load_brands,
    load_cosme_categories,
    load_rakuten_genres,
    PROJECT
)
from src.schema import (
    get_connection, create_schema, seed_sources,
    get_schema_info, ALL_DDL, DDL_INDEXES, DB_PATH,
)
from src.ingest import (
    ingest_rakuten_product,
    ingest_cosme_product,
    ingest_amazon_product,
    ingest_amazon_review,
    batch_ingest_cosme_reviews,
    ingest_trends_dataframe,
    ingest_yt_video,
    batch_ingest_yt_comments,
    upsert_category,
    normalise_date,
)

load_env()
print_ingestion_summary()
print()
print(f"Database path  : {DB_PATH}")
print(f"SQLite version : {sqlite3.sqlite_version}")
print()

# Verify raw data directories
dirs = {
    "Rakuten products" : DATA_RAW / "rakuten" / "products",
    "Rakuten ranking"  : DATA_RAW / "rakuten" / "ranking",
    "cosme products"   : DATA_RAW / "cosme"   / "products",
    "cosme reviews"    : DATA_RAW / "cosme"   / "reviews",
    "Amazon products"  : DATA_RAW / "amazon"  / "products",
    "Amazon reviews"   : DATA_RAW / "amazon"  / "reviews",
    "YouTube"          : DATA_RAW / "youtube",
    "YouTube comments" : DATA_RAW / "youtube" / "comments",
    "Trends"           : DATA_RAW / "trends",
}
print("Raw data inventory:")
for name, path in dirs.items():
    if path.exists():
        files = list(path.glob("*.json")) if path.is_dir() else []
        print(f"  OK  {name:<22} {len(files)} files")
    else:
        print(f"  !!  {name:<22} NOT FOUND")


17:57:37  INFO      .env loaded from C:\Users\stanl\OneDrive\Desktop\VSCode_Projects\6. Beauty_ConsumerPulse\.env


SIGNAL/PULSE — INGESTION SCOPE
  Project        : Signal/Pulse v2.0
  Time range     : 2020-01-01 → 2025-12-31
  Baseline       : 2019-01-01 (pre-COVID)
  Anchor term    : スキンケア
  Layer 2 top N  : 50 products per category

  @cosme — Layer 1 sweep  : 11 categories
  @cosme — Layer 2 corpus : 10 categories
  Trend seed terms        : 21
  Exclusions              : 10 terms flagged

Database path  : C:\Users\stanl\OneDrive\Desktop\VSCode_Projects\6. Beauty_ConsumerPulse\data\signal_pulse.db
SQLite version : 3.51.2

Raw data inventory:
  OK  Rakuten products       77 files
  OK  Rakuten ranking        77 files
  OK  cosme products         11 files
  OK  cosme reviews          150 files
  OK  Amazon products        1 files
  OK  Amazon reviews         111 files
  OK  YouTube                2 files
  OK  YouTube comments       225 files
  OK  Trends                 70 files


## 1. Create Schema

In [2]:
conn = get_connection()

print("Creating tables:")
for table_name, ddl in ALL_DDL:
    conn.execute(ddl)
    conn.commit()
    print(f"  OK  {table_name}")

print()
print("Creating indexes:")
for idx_ddl in DDL_INDEXES:
    conn.execute(idx_ddl)
    idx_name = idx_ddl.split("idx_")[1].split(" ")[0]
    print(f"  OK  idx_{idx_name}")
conn.commit()


Creating tables:
  OK  sources
  OK  categories
  OK  brands
  OK  products
  OK  reviewers
  OK  reviews
  OK  trends_weekly
  OK  yt_videos
  OK  yt_comments

Creating indexes:
  OK  idx_reviews_date
  OK  idx_reviews_year_month
  OK  idx_reviews_category
  OK  idx_reviews_brand
  OK  idx_reviews_source
  OK  idx_reviews_reviewer
  OK  idx_trends_term_week
  OK  idx_trends_week
  OK  idx_ytcomments_date
  OK  idx_ytcomments_year
  OK  idx_ytcomments_video
  OK  idx_products_category
  OK  idx_products_brand
  OK  idx_products_snapshot


## 2. Seed Sources, Brands, Categories

In [3]:
# Sources
seed_sources(conn)
df_sources = pd.read_sql("SELECT * FROM sources", conn)
print("Sources:")
print(df_sources.to_string(index=False))
print()

# Brands
all_brands = load_brands(group="all")
inserted_b = 0
for _, row in all_brands.iterrows():
    conn.execute(
        "INSERT OR IGNORE INTO brands "
        "(brand_name_jp, brand_name_en, parent_company, brand_group, is_target, tier) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (row["brand_name_jp"], row.get("brand_name_en"),
         row.get("parent_company"), row.get("brand_group"),
         int(row.get("is_target", 0)), row.get("tier"))
    )
    if conn.total_changes > inserted_b:
        inserted_b += 1
conn.commit()
print(f"Brands seeded: {inserted_b}")
print()

# @cosme categories
cosme_cats = load_cosme_categories(layer1=True)
cat_ids = {}
for _, row in cosme_cats.iterrows():
    internal_id = upsert_category(
        conn,
        source_id=2,
        source_cat_id=row["cosme_category_id"],
        source_cat_name=row["category_name_jp"],
        normalized_name=row["normalized_name"],
        tier=row["tier"],
    )
    cat_ids[str(row["cosme_category_id"])] = internal_id
conn.commit()
print(f"@cosme categories seeded: {len(cat_ids)}")

# Rakuten genres
rakuten_genres = load_rakuten_genres(layer1=True)
rak_cat_ids = {}
for _, row in rakuten_genres.iterrows():
    internal_id = upsert_category(
        conn,
        source_id=1,
        source_cat_id=str(row["rakuten_genre_id"]),
        source_cat_name=row["genre_name_jp"],
        normalized_name=row["normalized_name"],
        tier=row["tier"] if row["tier"] in ("skincare", "cosmetics", "other") else "other",
    )
    rak_cat_ids[str(row["rakuten_genre_id"])] = internal_id
conn.commit()
print(f"Rakuten genres seeded  : {len(rak_cat_ids)}")


Sources:
 source_id   source_name                                         description first_loaded_at
         1       rakuten    Rakuten Ichiba API — current commercial snapshot            None
         2         cosme              @cosme — primary review/customer layer            None
         3     amazon_jp Amazon.co.jp — secondary review layer (conditional)            None
         4 google_trends      Google Trends JP — weekly search demand signal            None
         5       youtube        YouTube JP — creator/influencer signal layer            None

Brands seeded: 51

@cosme categories seeded: 11
Rakuten genres seeded  : 11


## 3. Ingest — Rakuten

Date-stamped per-genre snapshots. Both products and rankings ingested.
`snapshot_date` preserved for temporal analysis across multiple snapshots.

In [4]:
rak_products_dir = DATA_RAW / "rakuten" / "products"
rak_ranking_dir  = DATA_RAW / "rakuten" / "ranking"

product_files = sorted(rak_products_dir.glob("rakuten_products_*.json"))
ranking_files = sorted(rak_ranking_dir.glob("rakuten_ranking_*.json"))

print(f"Rakuten product files : {len(product_files)}")
print(f"Rakuten ranking files : {len(ranking_files)}")
print()

total_inserted = 0

for fpath in product_files:
    # Filename: rakuten_products_{genre_id}_{date}.json
    parts        = fpath.stem.split("_")
    genre_id     = parts[2]
    snapshot_date = parts[3] if len(parts) > 3 else str(datetime.today().date())
    cat_internal = rak_cat_ids.get(genre_id)

    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    items = data if isinstance(data, list) else data.get("items", data.get("products", []))
    inserted = 0
    for i, wrapper in enumerate(items):
        item = wrapper.get("Item", wrapper)
        ok = ingest_rakuten_product(
            conn, item,
            category_id=cat_internal,
            snapshot_date=snapshot_date,
            ranking_position=i + 1,
        )
        if ok:
            inserted += 1

    conn.commit()
    print(f"  {fpath.name:<55} {inserted:>5} products")
    total_inserted += inserted

for fpath in ranking_files:
    parts         = fpath.stem.split("_")
    genre_id      = parts[2]
    snapshot_date = parts[3] if len(parts) > 3 else str(datetime.today().date())
    cat_internal  = rak_cat_ids.get(genre_id)

    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    items = data if isinstance(data, list) else data.get("Items", data.get("products", []))
    inserted = 0
    for i, wrapper in enumerate(items):
        item = wrapper.get("Item", wrapper)
        ok = ingest_rakuten_product(
            conn, item,
            category_id=cat_internal,
            snapshot_date=snapshot_date,
            ranking_position=i + 1,
            is_ranking=True,
        )
        if ok:
            inserted += 1
    conn.commit()
    total_inserted += inserted

count = conn.execute("SELECT COUNT(*) FROM products WHERE source_id=1").fetchone()[0]
print(f"\nRakuten total: {total_inserted} ingested | {count:,} in DB")


Rakuten product files : 77
Rakuten ranking files : 77



  rakuten_products_100939.json                             3000 products


  rakuten_products_100939_2026-04-02.json                    43 products


  rakuten_products_100939_2026-04-10.json                    29 products


  rakuten_products_100939_2026-04-17.json                    22 products


...

  rakuten_products_564517_2026-04-17.json                    98 products


  rakuten_products_564517_2026-04-25.json                    78 products


  rakuten_products_564517_2026-05-05.json                   125 products


  rakuten_products_564517_2026-05-12.json                   206 products



Rakuten total: 36255 ingested | 36,255 in DB


## 4. Ingest — @cosme Layer 1 (Product Metadata)

One row per product. `first_review_date` = oldest review seen — launch velocity proxy.

In [5]:
cosme_products_dir = DATA_RAW / "cosme" / "products"
product_files = sorted(cosme_products_dir.glob("cosme_products_*.json"))
print(f"@cosme product files: {len(product_files)}")
print()

total_inserted = 0
for fpath in product_files:
    cosme_cat_id = fpath.stem.split("_")[-1]
    cat_internal = cat_ids.get(str(cosme_cat_id))

    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    products = data.get("products", data) if isinstance(data, dict) else data
    inserted = 0
    for item in products:
        ok = ingest_cosme_product(conn, item, category_id=cat_internal)
        if ok:
            inserted += 1

    conn.commit()
    print(f"  {fpath.name:<45} {inserted:>4} products  (cat_id={cosme_cat_id})")
    total_inserted += inserted

count = conn.execute("SELECT COUNT(*) FROM products WHERE source_id=2").fetchone()[0]
print(f"\n@cosme products: {total_inserted} ingested | {count:,} in DB")


@cosme product files: 11

  cosme_products_1004.json                        50 products  (cat_id=1004)
  cosme_products_1005.json                        50 products  (cat_id=1005)
  cosme_products_1006.json                        50 products  (cat_id=1006)
  cosme_products_1037.json                        50 products  (cat_id=1037)
  cosme_products_800.json                         50 products  (cat_id=800)
  cosme_products_900.json                         50 products  (cat_id=900)
  cosme_products_901.json                         50 products  (cat_id=901)
  cosme_products_902.json                         50 products  (cat_id=902)
  cosme_products_912.json                         50 products  (cat_id=912)
  cosme_products_913.json                         50 products  (cat_id=913)
  cosme_products_916.json                         50 products  (cat_id=916)

@cosme products: 550 ingested | 470 in DB


## 5. Ingest — @cosme Layer 2 (Reviews)

Stratified sample: ~50 pages per product, step-sampled across full review history.
Each file contains one product with a `reviews` array and `year_dist` metadata.

In [6]:
cosme_reviews_dir = DATA_RAW / "cosme" / "reviews"
review_files = sorted(cosme_reviews_dir.glob("cosme_reviews_*.json"))
review_files = [f for f in review_files if ".done" not in f.name]
print(f"@cosme review files: {len(review_files)}")
print()

totals    = {"inserted": 0, "skipped": 0, "errors": 0}
cat_counts = {}

for fpath in review_files:
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    reviews = data.get("reviews", [])
    if not reviews:
        continue

    # Filename: cosme_reviews_{cat_id}_{pid}.json
    parts        = fpath.stem.split("_")
    cosme_cat_id = parts[2] if len(parts) > 2 else None
    cat_internal = cat_ids.get(str(cosme_cat_id)) if cosme_cat_id else None

    stats = batch_ingest_cosme_reviews(conn, reviews, category_id=cat_internal)
    for k in totals:
        totals[k] += stats[k]
    cat_counts[cosme_cat_id] = cat_counts.get(cosme_cat_id, 0) + stats["inserted"]

conn.commit()
count = conn.execute("SELECT COUNT(*) FROM reviews WHERE source_id=2").fetchone()[0]
print(f"Totals: {totals}")
print(f"@cosme reviews in DB: {count:,}")
print()
print("By category:")
for cat_id, n in sorted(cat_counts.items()):
    print(f"  cat {cat_id}: {n:,} reviews")


17:58:05  INFO      @cosme batch: {'inserted': 317, 'skipped': 0, 'errors': 0}


17:58:05  INFO      @cosme batch: {'inserted': 321, 'skipped': 0, 'errors': 0}


17:58:05  INFO      @cosme batch: {'inserted': 352, 'skipped': 0, 'errors': 0}


@cosme review files: 150



17:58:05  INFO      @cosme batch: {'inserted': 340, 'skipped': 0, 'errors': 0}


...

17:58:16  INFO      @cosme batch: {'inserted': 353, 'skipped': 0, 'errors': 0}


17:58:16  INFO      @cosme batch: {'inserted': 325, 'skipped': 0, 'errors': 0}


17:58:16  INFO      @cosme batch: {'inserted': 477, 'skipped': 0, 'errors': 0}


17:58:16  INFO      @cosme batch: {'inserted': 332, 'skipped': 0, 'errors': 0}


Totals: {'inserted': 45510, 'skipped': 5068, 'errors': 0}
@cosme reviews in DB: 45,510

By category:
  cat 1004: 5,573 reviews
  cat 1005: 4,521 reviews
  cat 1006: 5,053 reviews
  cat 1037: 3,464 reviews
  cat 900: 4,815 reviews
  cat 901: 4,902 reviews
  cat 902: 2,279 reviews
  cat 912: 5,073 reviews
  cat 913: 5,113 reviews
  cat 916: 4,717 reviews


In [7]:
# ── Reconcile @cosme product category_id with the canonical review path ──
# A product first catalogued under a broad Layer-1 ranking (e.g. skincare_all)
# keeps that generic category_id, while its Layer-2 reviews carry the specific
# category they were scraped under — so products.category_id and
# reviews.category_id disagree for the same records. reviews.category_id is the
# canonical (more specific) signal: align products to it so every join path
# through products yields the same tier. (Methodology audit Revision 3, Issue 2.)
n = conn.execute("""
    UPDATE products
    SET category_id = (
        SELECT r.category_id FROM reviews r
        WHERE r.product_id = products.product_id
        GROUP BY r.category_id ORDER BY COUNT(*) DESC LIMIT 1
    )
    WHERE source_id = 2
      AND EXISTS (SELECT 1 FROM reviews r2 WHERE r2.product_id = products.product_id)
      AND category_id != (
        SELECT r.category_id FROM reviews r
        WHERE r.product_id = products.product_id
        GROUP BY r.category_id ORDER BY COUNT(*) DESC LIMIT 1
      )
""").rowcount
conn.commit()
print(f"Reconciled {n} @cosme products to their dominant review category.")


Reconciled 19 @cosme products to their dominant review category.


In [8]:
import json
from pathlib import Path
from collections import Counter

review_dir = DATA_RAW / "cosme" / "reviews"
files = sorted(review_dir.glob("cosme_reviews_*.json"))
files = [f for f in files if ".done" not in f.name]

total = 0
has_date = 0
date_samples = []
none_samples = []

for fpath in files[:10]:  # check first 10 files
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    reviews = data.get("reviews", [])
    for r in reviews:
        total += 1
        d = r.get("review_date")
        if d:
            has_date += 1
            if len(date_samples) < 5:
                date_samples.append(d)
        else:
            if len(none_samples) < 3:
                # Show what other fields exist when date is None
                none_samples.append({k: v for k, v in r.items() 
                                     if k in ['review_date', 'scraped_date', 
                                              'reviewer_meta', 'source_review_id']})

print(f"Total reviews checked : {total}")
print(f"Have review_date      : {has_date} ({has_date/total*100:.1f}%)")
print(f"Missing review_date   : {total-has_date} ({(total-has_date)/total*100:.1f}%)")
print(f"\nDate samples: {date_samples}")
print(f"\nSample None reviews:")
for s in none_samples:
    print(f"  {s}")

Total reviews checked : 3800
Have review_date      : 3486 (91.7%)
Missing review_date   : 314 (8.3%)

Date samples: ['2026/5/15 23:59:44', '2026/5/15 10:42:25', '2026/5/14 23:19:45', '2026/5/13 15:11:34', '2026/5/11 09:06:34']

Sample None reviews:
  {'source_review_id': '516572019', 'review_date': None, 'reviewer_meta': ['49歳', '乾燥肌', 'クチコミ投稿2595件', '通報する', '参考にしたい！ありがとう'], 'scraped_date': '2026-05-17'}
  {'source_review_id': '516864581', 'review_date': None, 'reviewer_meta': ['36歳', '混合肌', 'クチコミ投稿72件', '200ml', '通報する', '参考にしたい！ありがとう'], 'scraped_date': '2026-05-17'}
  {'source_review_id': '516610146', 'review_date': None, 'reviewer_meta': ['39歳', '敏感肌', 'クチコミ投稿55件', '通報する', '参考にしたい！ありがとう'], 'scraped_date': '2026-05-17'}


## 6. Ingest — Amazon JP

Single-pass: metadata and reviews from same product page response.
`amazon_products.json` = consolidated Layer 1.
`amazon_reviews_{asin}.json` = per-product Layer 2.

In [9]:
# Layer 1 — consolidated product file
l1_path = DATA_RAW / "amazon" / "products" / "amazon_products.json"

if not l1_path.exists():
    print(f"Amazon Layer 1 not found: {l1_path}")
else:
    with open(l1_path, encoding="utf-8") as f:
        data = json.load(f)

    products    = data.get("products", data) if isinstance(data, dict) else data
    inserted_l1 = 0
    for item in products:
        ok = ingest_amazon_product(conn, item)
        if ok:
            inserted_l1 += 1
    conn.commit()
    count = conn.execute("SELECT COUNT(*) FROM products WHERE source_id=3").fetchone()[0]
    print(f"Amazon Layer 1: {inserted_l1} ingested | {count} in DB")
    print()

# Layer 2 — per-ASIN review files
amazon_reviews_dir = DATA_RAW / "amazon" / "reviews"
review_files       = sorted(amazon_reviews_dir.glob("amazon_reviews_*.json"))
print(f"Amazon review files: {len(review_files)}")

totals_amz = {"inserted": 0, "skipped": 0, "errors": 0}
for fpath in review_files:
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    reviews = data.get("reviews", [])
    for rev in reviews:
        try:
            ok = ingest_amazon_review(conn, rev)
            totals_amz["inserted" if ok else "skipped"] += 1
        except Exception as e:
            totals_amz["errors"] += 1

conn.commit()
count_amz = conn.execute("SELECT COUNT(*) FROM reviews WHERE source_id=3").fetchone()[0]
print(f"Amazon reviews: {totals_amz}")
print(f"Amazon reviews in DB: {count_amz:,}")


Amazon Layer 1: 111 ingested | 111 in DB

Amazon review files: 111


Amazon reviews: {'inserted': 1079, 'skipped': 10, 'errors': 0}
Amazon reviews in DB: 1,079


## 7. Ingest — YouTube

Video metadata from `youtube_video_stats.json` (includes `search_category`).
Comments from `comments/youtube_comments_{vid_id}.json`.

In [10]:
yt_dir       = DATA_RAW / "youtube"
comments_dir = yt_dir / "comments"
stats_path   = yt_dir / "youtube_video_stats.json"

if not stats_path.exists():
    print(f"YouTube stats not found: {stats_path}")
else:
    with open(stats_path, encoding="utf-8") as f:
        vlist = json.load(f)

    inserted_v = 0
    for v in vlist:
        ok = ingest_yt_video(conn, {
            "yt_video_id":     v.get("video_id", v.get("yt_video_id")),
            "channel_name":    v.get("channel"),
            "title":           v.get("title"),
            "published_at":    v.get("published_at"),
            "view_count":      v.get("view_count"),
            "like_count":      v.get("like_count"),
            "comment_count":   v.get("comment_count"),
            "search_category": v.get("search_category", ""),
        })
        if ok:
            inserted_v += 1
    conn.commit()
    print(f"YouTube videos: {inserted_v} ingested")

comment_files  = sorted(comments_dir.glob("youtube_comments_*.json"))
print(f"YouTube comment files: {len(comment_files)}")

total_comments = 0
errors_c       = 0
for fpath in comment_files:
    yt_vid_id = fpath.stem.replace("youtube_comments_", "")
    row = conn.execute(
        "SELECT video_id FROM yt_videos WHERE yt_video_id=?", (yt_vid_id,)
    ).fetchone()
    if not row:
        continue

    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    comments   = data.get("comments", data) if isinstance(data, dict) else data
    normalised = []
    for c in comments:
        normalised.append({
            "yt_comment_id": c.get("yt_comment_id", c.get("comment_id")),
            "text":          c.get("text", c.get("comment_text", "")),
            "published_at":  c.get("published_at"),
            "like_count":    c.get("like_count", 0),
            "is_reply":      int(c.get("is_reply", False)),
            "is_japanese":   int(c.get("is_japanese", True)),
        })

    try:
        batch_ingest_yt_comments(conn, normalised, row[0])
        total_comments += len(normalised)
    except Exception as e:
        errors_c += 1

conn.commit()
vcount = conn.execute("SELECT COUNT(*) FROM yt_videos").fetchone()[0]
ccount = conn.execute("SELECT COUNT(*) FROM yt_comments").fetchone()[0]
print(f"YouTube: {vcount} videos | {ccount:,} comments in DB")
if errors_c:
    print(f"  Errors: {errors_c}")


17:58:35  INFO      YouTube comments batch: {'inserted': 66, 'skipped': 0, 'errors': 0}


17:58:35  INFO      YouTube comments batch: {'inserted': 641, 'skipped': 1, 'errors': 0}


17:58:35  INFO      YouTube comments batch: {'inserted': 278, 'skipped': 0, 'errors': 0}


17:58:35  INFO      YouTube comments batch: {'inserted': 100, 'skipped': 0, 'errors': 0}


17:58:35  INFO      YouTube comments batch: {'inserted': 185, 'skipped': 0, 'errors': 0}


...

17:58:41  INFO      YouTube comments batch: {'inserted': 302, 'skipped': 0, 'errors': 0}


17:58:41  INFO      YouTube comments batch: {'inserted': 980, 'skipped': 0, 'errors': 0}


17:58:41  INFO      YouTube comments batch: {'inserted': 458, 'skipped': 1, 'errors': 0}


17:58:41  INFO      YouTube comments batch: {'inserted': 388, 'skipped': 1, 'errors': 0}


YouTube: 248 videos | 60,676 comments in DB


## 8. Ingest — Google Trends

Block A (unanchored) and Block B (anchored) ingested into `trends_weekly`.
Block C (Rising queries) stored as raw JSON — loaded directly in NB06.

In [11]:
trends_dir = DATA_RAW / "trends"

block_files = {
    "A": sorted(trends_dir.glob("trends_A_*.json")),
    "B": sorted(trends_dir.glob("trends_B_*.json")),
    "C": sorted(trends_dir.glob("trends_C_*.json")),
}

for block, files in block_files.items():
    print(f"Block {block}: {len(files)} files")
print()

total_rows = 0

for block, files in block_files.items():
    # Block C = Rising queries — not time-series, skip for trends_weekly
    if block == "C":
        print(f"Block C: {len(files)} Rising query files — loaded in NB06, skipping here")
        continue

    block_rows = 0
    for fpath in files:
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)

        try:
            if isinstance(data, list):
                df = pd.DataFrame(data)
            elif isinstance(data, dict):
                df = pd.DataFrame(data)
                if all(re.match(r"\d{4}-\d{2}-\d{2}", str(i)) for i in list(df.index)[:3]):
                    df = df.reset_index().rename(columns={"index": "date"})
                    df = df.melt(id_vars="date", var_name="term", value_name="value")
                else:
                    df = df.T.reset_index().rename(columns={"index": "date"})
                    df = df.melt(id_vars="date", var_name="term", value_name="value")

            if "date" not in df.columns:
                print(f"  SKIP {fpath.name} — no date column")
                continue

            df["date"] = pd.to_datetime(df["date"])
            df = df.set_index("date")
            n = ingest_trends_dataframe(conn, df, term_group=f"block_{block}")
            block_rows += n
        except Exception as e:
            print(f"  ERROR {fpath.name}: {e}")

    conn.commit()
    print(f"Block {block}: {block_rows:,} rows ingested")
    total_rows += block_rows

total_tw = conn.execute("SELECT COUNT(*) FROM trends_weekly").fetchone()[0]
print(f"\ntrends_weekly total: {total_tw:,} rows")


17:58:41  INFO      Trends ingested: 87 rows for 1 terms


17:58:41  INFO      Trends ingested: 87 rows for 1 terms


17:58:41  INFO      Trends ingested: 87 rows for 1 terms


17:58:41  INFO      Trends ingested: 87 rows for 1 terms


17:58:41  INFO      Trends ingested: 87 rows for 1 terms


...

17:58:41  INFO      Trends ingested: 87 rows for 1 terms


17:58:42  INFO      Trends ingested: 435 rows for 5 terms


17:58:42  INFO      Trends ingested: 348 rows for 5 terms


17:58:42  INFO      Trends ingested: 348 rows for 5 terms


Block A: 1,740 rows ingested
Block B: 1,131 rows ingested
Block C: 43 Rising query files — loaded in NB06, skipping here

trends_weekly total: 2,871 rows


## 9. Quality Checks

In [12]:
info = get_schema_info(conn)
print("DATABASE SUMMARY")
print("=" * 56)
print(f"{'Table':<24} {'Rows':>8}  {'Cols':>5}  {'FKs':>4}")
print("-" * 56)
for t in info:
    print(f"  {t['table']:<22} {t['rows']:>8,}  {t['columns']:>5}  {t['foreign_keys']:>4}")


DATABASE SUMMARY
Table                        Rows   Cols   FKs
--------------------------------------------------------
  brands                       51      8     0
  categories                   22      8     2
  products                 36,836     14     3
  reviewers                35,995      6     1
  reviews                  46,589     19     5
  sources                       5      4     0
  trends_weekly             2,871      9     0
  yt_comments              60,676     11     1
  yt_videos                   248     13     1


In [13]:
total_rev = conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
if total_rev > 0:
    cols = ["review_date", "review_year", "review_month",
            "rating", "review_text", "category_id", "reviewer_id"]
    print(f"NULL AUDIT — reviews ({total_rev:,} rows)")
    print("-" * 52)
    for col in cols:
        try:
            n   = conn.execute(f"SELECT COUNT(*) FROM reviews WHERE {col} IS NULL").fetchone()[0]
            pct = n / total_rev * 100
            flag = "WARN" if pct > 20 else "OK  "
            print(f"  [{flag}] {col:<22} {n:>7,} nulls ({pct:5.1f}%)")
        except Exception as e:
            print(f"  [ERR ] {col}: {e}")


NULL AUDIT — reviews (46,589 rows)
----------------------------------------------------
  [OK  ] review_date              3,973 nulls (  8.5%)
  [OK  ] review_year              3,973 nulls (  8.5%)
  [OK  ] review_month             3,973 nulls (  8.5%)
  [OK  ] rating                     199 nulls (  0.4%)
  [OK  ] review_text                  0 nulls (  0.0%)
  [OK  ] category_id              1,079 nulls (  2.3%)
  [OK  ] reviewer_id                239 nulls (  0.5%)


In [14]:
if conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0] > 0:
    sql = (
        "SELECT review_year, COUNT(*) AS review_count, "
        "COUNT(DISTINCT reviewer_id) AS unique_reviewers, "
        "ROUND(AVG(rating), 2) AS avg_rating, "
        "SUM(CASE WHEN LENGTH(COALESCE(review_text,''))>20 THEN 1 ELSE 0 END) AS with_text, "
        "COUNT(DISTINCT source_id) AS sources "
        "FROM reviews WHERE review_year IS NOT NULL "
        "GROUP BY review_year ORDER BY review_year"
    )
    df_cov = pd.read_sql(sql, conn)
    print("REVIEW YEAR COVERAGE:")
    print(df_cov.to_string(index=False))
    years = set(df_cov["review_year"].tolist())
    print()
    for y in [2020, 2021, 2022]:
        flag = "YES" if y in years else "MISSING — check scrape depth"
        print(f"  {y}: {flag}")


REVIEW YEAR COVERAGE:
 review_year  review_count  unique_reviewers  avg_rating  with_text  sources
        2005             4                 3        3.00          4        1
        2006             2                 2        4.33          2        1
        2007             2                 2        2.67          2        1
        2008            11                 9        2.82         11        1
        2009             8                 7        4.33          8        1
        2010             5                 3        3.93          5        1
        2011            15                12        3.71         15        1
        2012             8                 8        3.67          8        1
        2013            15                14        3.71         15        1
        2014            17                17        3.12         17        1
        2015            28                26        3.71         28        1
        2016            30                28        3.

In [15]:
# ── YouTube quality audit ─────────────────────────────────────────────────
# Audits yt_videos + yt_comments on every run.
# Known design decisions documented inline so findings are self-explanatory.

print("YOUTUBE QUALITY AUDIT")
print("=" * 60)

total_c = conn.execute("SELECT COUNT(*) FROM yt_comments").fetchone()[0]
total_v = conn.execute("SELECT COUNT(*) FROM yt_videos").fetchone()[0]
print(f"  yt_videos   : {total_v:,}")
print(f"  yt_comments : {total_c:,}")
print()

# ── 1. Null audit — yt_comments ───────────────────────────────────────────
print("NULL AUDIT — yt_comments")
print("-" * 60)
for col in ["comment_text", "published_at", "is_japanese", "is_reply"]:
    n   = conn.execute(
        f"SELECT COUNT(*) FROM yt_comments WHERE {col} IS NULL"
    ).fetchone()[0]
    pct  = n / total_c * 100
    flag = "WARN" if pct > 5 else "OK  "
    print(f"  [{flag}] {col:<20} {n:>6,} nulls ({pct:.1f}%)")

# ── 2. Reply rate — design note ───────────────────────────────────────────
print()
replies = conn.execute(
    "SELECT COUNT(*) FROM yt_comments WHERE is_reply = 1"
).fetchone()[0]
print(f"  Reply comments : {replies:,} ({replies/total_c*100:.1f}% of total)")
print(f"  DESIGN NOTE    : NB01e collected topLevelComment only — is_reply")
print(f"                   is always 0 by construction, not a data gap.")
print(f"                   All 60,676 comments are top-level. Safe for TF-IDF.")

# ── 3. Zero-collection videos (API count > 0 but nothing ingested) ────────
print()
print("ZERO-COLLECTION VIDEOS")
print("-" * 60)
df_zero = pd.read_sql("""
    SELECT v.channel_name, v.search_category, v.comment_count
    FROM yt_videos v
    LEFT JOIN yt_comments c USING (video_id)
    WHERE v.comment_count > 0
    GROUP BY v.video_id
    HAVING COUNT(c.comment_id) = 0
""", conn)
if len(df_zero):
    print(f"  WARNING — {len(df_zero)} video(s) with comments in API but 0 collected:")
    print(df_zero.to_string(index=False))
    print(f"  Likely cause: comments disabled or private between scrape and ingestion.")
    print(f"  Impact: minor — check search_category for analytical weight.")
else:
    print("  OK — all videos with API comment_count > 0 have collected comments.")

# ── 4. Truncation risk (capped at 1,000 per video in NB01e) ──────────────
print()
print("TRUNCATION RISK — videos where comment_count > 1,000")
print("-" * 60)
df_trunc = pd.read_sql("""
    SELECT v.channel_name, v.search_category,
           v.comment_count                  AS api_count,
           COUNT(c.comment_id)              AS collected,
           v.comment_count - COUNT(c.comment_id) AS gap
    FROM yt_videos v
    LEFT JOIN yt_comments c USING (video_id)
    GROUP BY v.video_id
    HAVING v.comment_count > 1000
    ORDER BY gap DESC
    LIMIT 10
""", conn)
if len(df_trunc):
    print(df_trunc.to_string(index=False))
    print()
    print("  NOTE: NB01e cap = 1,000 comments/video (10 pages × 100).")
    print("  High-engagement videos are undersampled relative to their reach.")
    print("  TF-IDF treats all comments equally regardless of video view_count.")
else:
    print("  OK — no videos exceed the 1,000 comment cap.")

# ── 5. Temporal coverage ──────────────────────────────────────────────────
print()
print("TEMPORAL COVERAGE — yt_comments by year")
print("-" * 60)
df_yr = pd.read_sql("""
    SELECT strftime('%Y', published_at) AS yr, COUNT(*) AS n_comments
    FROM yt_comments
    WHERE published_at IS NOT NULL
    GROUP BY yr ORDER BY yr
""", conn)
print(df_yr.to_string(index=False))
print()
print("  NOTE: 2026 comments (Jan–Mar scrape window) are in the DB.")
print("  Dashboard volume chart currently filters BETWEEN 2019 AND 2025.")
print("  2026 contributes to TF-IDF but is invisible in trend charts.")
print("  Consider including as partial year to show trajectory continuing.")

# ── 6. search_category completeness ──────────────────────────────────────
print()
print("SEARCH_CATEGORY DISTRIBUTION — yt_videos")
print("-" * 60)
df_cat = pd.read_sql("""
    SELECT search_category, COUNT(*) AS videos
    FROM yt_videos
    WHERE search_category IS NOT NULL AND search_category != ''
    GROUP BY search_category ORDER BY videos DESC
""", conn)
print(df_cat.to_string(index=False))
no_cat = conn.execute(
    "SELECT COUNT(*) FROM yt_videos WHERE search_category IS NULL OR search_category = ''"
).fetchone()[0]
if no_cat:
    print(f"\n  WARNING: {no_cat} videos missing search_category.")


YOUTUBE QUALITY AUDIT
  yt_videos   : 248
  yt_comments : 60,676

NULL AUDIT — yt_comments
------------------------------------------------------------
  [OK  ] comment_text              0 nulls (0.0%)
  [OK  ] published_at              0 nulls (0.0%)
  [OK  ] is_japanese               0 nulls (0.0%)
  [OK  ] is_reply                  0 nulls (0.0%)

  Reply comments : 0 (0.0% of total)
  DESIGN NOTE    : NB01e collected topLevelComment only — is_reply
                   is always 0 by construction, not a data gap.
                   All 60,676 comments are top-level. Safe for TF-IDF.

ZERO-COLLECTION VIDEOS
------------------------------------------------------------
  WARNING — 25 video(s) with comments in API but 0 collected:
                 channel_name search_category  comment_count
               五彩緋夏 ひなちゃん5しゃい          口紅・リップ           1039
                        水越みさと          口紅・リップ            647
                        水越みさと          アイシャドウ            375
                 

  yr  n_comments
2019         559
2020        1531
2021        4355
2022        6537
2023       10230
2024       18455
2025       17326
2026        1683

  NOTE: 2026 comments (Jan–Mar scrape window) are in the DB.
  Dashboard volume chart currently filters BETWEEN 2019 AND 2025.
  2026 contributes to TF-IDF but is invisible in trend charts.
  Consider including as partial year to show trajectory continuing.

SEARCH_CATEGORY DISTRIBUTION — yt_videos
------------------------------------------------------------
search_category  videos
            美容液      20
        敏感肌・乾燥肌      20
          成分・知識      20
            化粧水      20
        乳液・クリーム      20
       メンズ・プチプラ      20
       ファンデーション      20
        エイジングケア      20
         アイシャドウ      20
         その他メイク      20
         口紅・リップ      15
          日焼け止め      10
          韓国コスメ       9
            洗顔料       8
         ニキビ・毛穴       6


In [16]:
sql_src = (
    "SELECT s.source_name, COUNT(*) as reviews "
    "FROM reviews r JOIN sources s USING (source_id) "
    "GROUP BY s.source_name"
)
df_src = pd.read_sql(sql_src, conn)
print("Reviews by source:")
print(df_src.to_string(index=False))
print()

sql_yt = (
    "SELECT search_category, COUNT(*) as videos FROM yt_videos "
    "WHERE search_category IS NOT NULL AND search_category != '' "
    "GROUP BY search_category ORDER BY videos DESC"
)
df_yt = pd.read_sql(sql_yt, conn)
if len(df_yt) > 0:
    print("YouTube videos by search_category:")
    print(df_yt.to_string(index=False))


Reviews by source:
source_name  reviews
  amazon_jp     1079
      cosme    45510

YouTube videos by search_category:
search_category  videos
            美容液      20
        敏感肌・乾燥肌      20
          成分・知識      20
            化粧水      20
        乳液・クリーム      20
       メンズ・プチプラ      20
       ファンデーション      20
        エイジングケア      20
         アイシャドウ      20
         その他メイク      20
         口紅・リップ      15
          日焼け止め      10
          韓国コスメ       9
            洗顔料       8
         ニキビ・毛穴       6


In [17]:
if conn.execute("SELECT COUNT(*) FROM trends_weekly").fetchone()[0] > 0:
    sql = (
        "SELECT term, term_group, "
        "MIN(week_start) AS first_week, MAX(week_start) AS last_week, "
        "COUNT(*) AS weeks "
        "FROM trends_weekly "
        "GROUP BY term ORDER BY term_group, term"
    )
    df_tw = pd.read_sql(sql, conn)
    print("GOOGLE TRENDS COVERAGE:")
    print(df_tw.to_string(index=False))


GOOGLE TRENDS COVERAGE:
    term term_group first_week  last_week  weeks
  アイシャドウ    block_A 2019-01-01 2026-03-01    174
  アゼライン酸    block_A 2019-01-01 2026-03-01     87
  エクソソーム    block_A 2019-01-01 2026-03-01     87
  グルタチオン    block_A 2019-01-01 2026-03-01     87
   スキンケア    block_A 2019-01-01 2026-03-01    174
    セラミド    block_A 2019-01-01 2026-03-01    174
 トラネキサム酸    block_A 2019-01-01 2026-03-01     87
ナイアシンアミド    block_A 2019-01-01 2026-03-01    174
  ヒアルロン酸    block_A 2019-01-01 2026-03-01    174
ビタミンC 美容    block_A 2019-01-01 2026-03-01     87
ファンデーション    block_A 2019-01-01 2026-03-01    174
   レチナール    block_A 2019-01-01 2026-03-01     87
   レチノール    block_A 2019-01-01 2026-03-01    174
      乳液    block_A 2019-01-01 2026-03-01    174
     化粧品    block_A 2019-01-01 2026-03-01    174
     化粧水    block_A 2019-01-01 2026-03-01    174
      口紅    block_A 2019-01-01 2026-03-01    174
   日焼け止め    block_A 2019-01-01 2026-03-01     87
      洗顔    block_A 2019-01-01 2026-03-01    

## 10. Close

In [18]:
conn.close()
print("=" * 60)
print("NB02 COMPLETE")
print("=" * 60)
print(f"Database: {DB_PATH}")
print()
print("Next: NB03 — SQL Analytical Foundation")
print("  Confirmatory shift analysis: skincare vs cosmetics 2020-2025")


NB02 COMPLETE
Database: C:\Users\stanl\OneDrive\Desktop\VSCode_Projects\6. Beauty_ConsumerPulse\data\signal_pulse.db

Next: NB03 — SQL Analytical Foundation
  Confirmatory shift analysis: skincare vs cosmetics 2020-2025
